MNIST as a regression problem. CNN + DNN with one output neuron

CJ Nov 2023

In [ ]:
from keras import layers, models, callbacks, optimizers
from keras.datasets import mnist
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  1


In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Oct 12 17:33:13 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.94                 Driver Version: 560.94         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A2000 8GB Lap...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   52C    P5              8W /   49W |    6932MiB /   8192MiB |     10%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Load and preprocess the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=42)

# Normalize pixel values to be between 0 and 1
x_train, x_val, x_test = x_train / 255.0, x_val / 255.0, x_test / 255.0

# Flatten the images, not needed for CNNs
#x_train = x_train.reshape((x_train.shape[0], -1))
#x_val = x_val.reshape((x_val.shape[0], -1))
#x_test = x_test.reshape((x_test.shape[0], -1))

In [ ]:
# Define the model (CNN+DNN)
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)  # One output neuron for regression
])

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

# Train the model with callback
cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=4,
                            restore_best_weights=True),
]
model.fit(x_train, y_train, epochs=20, batch_size=32,
          validation_data=(x_val, y_val),
          callbacks=cb_list,
          verbose=1)

Epoch 1/20
1688/1688 [==============================] - 7s 4ms/step - loss: 1.7210 - mae: 0.9032 - val_loss: 0.7890 - val_mae: 0.5979
Epoch 2/20
1688/1688 [==============================] - 6s 4ms/step - loss: 0.6353 - mae: 0.5340 - val_loss: 0.6637 - val_mae: 0.5567
Epoch 3/20
1688/1688 [==============================] - 6s 4ms/step - loss: 0.4684 - mae: 0.4472 - val_loss: 0.4267 - val_mae: 0.4015
Epoch 4/20
1688/1688 [==============================] - 6s 4ms/step - loss: 0.3766 - mae: 0.3933 - val_loss: 0.3890 - val_mae: 0.3802
Epoch 5/20
1688/1688 [==============================] - 6s 4ms/step - loss: 0.3164 - mae: 0.3544 - val_loss: 0.3233 - val_mae: 0.3383
Epoch 6/20
1688/1688 [==============================] - 6s 4ms/step - loss: 0.2723 - mae: 0.3276 - val_loss: 0.3156 - val_mae: 0.3265
Epoch 7/20
1688/1688 [==============================] - 6s 4ms/step - loss: 0.2405 - mae: 0.3046 - val_loss: 0.3022 - val_mae: 0.3261
Epoch 8/20
1688/1688 [==============================] - 6s 4ms

In [ ]:
# Evaluate the model on the test set
test_loss, test_mae = model.evaluate(x_test, y_test)
print(f'Test Mean Squared Error: {test_loss}, Test Mean Absolute Error: {test_mae}')

313/313 [==============================] - 1s 2ms/step - loss: 0.1566 - mae: 0.1648
Test Mean Squared Error: 0.156632661819458, Test Mean Absolute Error: 0.16480229794979095


In [ ]:
print(y_test)

[7 2 1 ... 4 5 6]


In [ ]:
#x_test[0].shape

In [ ]:
#print(x_test[0:2])

In [ ]:
print(model.predict(np.array([x_test[0]]))) # should be 7


1/1 [==============================] - 0s 56ms/step
[[7.0265393]]


In [ ]:
print(model.predict([x_test[:3]])) # First 3, shoud be 7 2 1

1/1 [==============================] - 0s 60ms/step
[[7.0265384]
 [1.8961005]
 [1.0209713]]


In [ ]:
print(model.predict(np.array([x_test[-1]]))) # last item. Should be 6

1/1 [==============================] - 0s 21ms/step
[[6.1446395]]


In [ ]:
print(model.predict([x_test[-3:]])) # last 3 items. Should be 4, 5, 6

1/1 [==============================] - 0s 31ms/step
[[3.8827739]
 [5.217829 ]
 [6.1446395]]


In [ ]:
# Python test
t=[1,2,3,4,5,6,7,8,9]
print(t[:3]+t[-3:])

[1, 2, 3, 7, 8, 9]


In [ ]:
#first three and last three items
predictions = model.predict([ np.concatenate((x_test[:3], x_test[-3:]), axis=0) ])
print(np.round(predictions))

1/1 [==============================] - 0s 31ms/step
[[7.]
 [2.]
 [1.]
 [4.]
 [5.]
 [6.]]
